# Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

df = yf.download("AAPL", start="2020-01-01", end="2026-07-01")

[*********************100%***********************]  1 of 1 completed


# Data exploration

In [ ]:
head = df.head(5)
info = df.info()
describe = df.describe()
null = df.isnull().sum()
print(null)

# Data cleaning and preprocessing

In [2]:
# Ensure the time series is sorted and cast to a datetime index, then handle missing values.
df = df.sort_index()
df.index = pd.to_datetime(df.index)

missing_counts = df.isna().sum()
print("Missing values per column:")
print(missing_counts)

# Use row-based dropna to preserve the core time series if there are any empty rows.
df = df.dropna()
print(f"\nRows after dropna: {len(df)}")

Missing values per column:
Price   Ticker
Close   AAPL      0
High    AAPL      0
Low     AAPL      0
Open    AAPL      0
Volume  AAPL      0
dtype: int64

Rows after dropna: 1631


# Data transformation using pandas

In [ ]:
# Create new analytical features from the raw price data.
df["Daily Change"] = df["Close"] - df["Open"]
df["Daily Pct Change"] = (df["Close"] - df["Open"]) / df["Open"] * 100
df["Daily Range"] = df["High"] - df["Low"]
df["Log Return"] = np.log(df["Close"] / df["Close"].shift(1))

# Aggregate at the monthly level to summarize performance.
monthly_summary = (
    df[["Close", "Daily Pct Change", "Daily Range"]].resample("ME")
    .agg(
        monthly_close=("Close", "last"),
        avg_pct_change=("Daily Pct Change", "mean"),
        avg_daily_range=("Daily Range", "mean"),
    )
)
monthly_summary["monthly_return"] = monthly_summary["monthly_close"].pct_change() * 100

print("Sample engineered features:")
display(df[["Daily Change", "Daily Pct Change", "Daily Range", "Log Return"]].head())
print("\nMonthly summary sample:")
display(monthly_summary.tail(5))

print("Top 5 daily percent gain days in 2024:")
display(df.loc["2024"].nlargest(5, "Daily Pct Change")[['Close', 'Daily Pct Change']])

ValueError: Invalid frequency: M. Failed to parse with error message: ValueError("'M' is no longer supported for offsets. Please use 'ME' instead.")

# Time series analysis

In [ ]:
# Compute rolling moving averages, monthly returns, and volatility.
df["MA_7"] = df["Close"].rolling(7, min_periods=1).mean()
df["MA_30"] = df["Close"].rolling(30, min_periods=1).mean()
df["Volatility_30d"] = df["Log Return"].rolling(30).std() * np.sqrt(252)

monthly_close = df["Close"].resample("M").last()
monthly_returns = monthly_close.pct_change() * 100

print("Rolling averages and volatility sample:")
display(df[["Close", "MA_7", "MA_30", "Volatility_30d"]].tail())
print("\nMonthly returns sample:")
display(monthly_returns.tail())

# Data visualization

In [ ]:
# Plot closing price with moving averages, monthly returns, and volatility metrics.
plt.figure(figsize=(12, 6))
plt.plot(df["Close"], label="Close", alpha=0.75)
plt.plot(df["MA_7"], label="7-day MA", linewidth=1)
plt.plot(df["MA_30"], label="30-day MA", linewidth=1)
plt.title("AAPL Close Price with Rolling Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price USD")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 4))
monthly_returns.plot(kind="bar", color="steelblue")
plt.title("AAPL Monthly Returns (%)")
plt.xlabel("Month")
plt.ylabel("Return (%)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
df["Volatility_30d"].plot(color="darkorange")
plt.title("AAPL 30-Day Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.grid(True)
plt.tight_layout()
plt.show()